# KKBox Churn Prediction — Model Comparison

Rewritten as a proper step-by-step notebook (original was one long cell). Logic is
unchanged from the source notebook; only structure, plus three compatibility fixes
for modern pandas/scikit-learn (noted inline where they occur).

**Note on data:** running on synthetic data matching the real v2 schema
(`train_v2.csv`, `transactions_v2.csv`, `user_logs_v2.csv`, `members_v3.csv`,
`sample_submission_v2.csv`) — real Kaggle access wasn't available. Swap in the real
files with the same names and every cell below runs unchanged.


## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

train = pd.read_csv('train_v2.csv')
sample_submission = pd.read_csv('sample_submission_v2.csv')
transactions = pd.read_csv('transactions_v2.csv')
user_logs = pd.read_csv('user_logs_v2.csv')
members = pd.read_csv('members_v3.csv')

# set the options so the output format can be displayed correctly
pd.set_option('expand_frame_repr', True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

print(f"train: {train.shape}, transactions: {transactions.shape}, "
      f"user_logs: {user_logs.shape}, members: {members.shape}")

In [ ]:
# check the number of duplicate accounts in each table
print('train duplicates:', train.duplicated('msno').sum())
print('sample_submission duplicates:', sample_submission.duplicated('msno').sum())
print('transactions duplicates:', transactions.duplicated('msno').sum())
print('user_logs duplicates:', user_logs.duplicated('msno').sum())
print('members duplicates:', members.duplicated('msno').sum())

## 2. Aggregate Transactions and User Logs to One Row per User

Both `transactions` and `user_logs` have multiple rows per user (one per transaction /
per day). We collapse each to a single row per `msno` before merging into the modeling
table.

In [ ]:
# returns the max value of numerical variables and membership_expire_date
# returns the min value of transaction date
# returns the mode of ordinal variable and dummy variables, if multiple values share the same frequency, keep the first one
transactions_v2 = transactions.groupby('msno', as_index=False).agg({
    'payment_method_id': lambda x: x.value_counts().index[0],
    'payment_plan_days': 'max',
    'plan_list_price': 'max',
    'actual_amount_paid': 'max',
    'is_auto_renew': lambda x: x.value_counts().index[0],
    'transaction_date': 'min',
    'membership_expire_date': 'max',
    'is_cancel': lambda x: x.value_counts().index[0],
})
transactions_v2.head()

In [ ]:
# returns the max value of date and number of unique songs
# returns the sum of other variables
user_logs_v2 = user_logs.groupby('msno', as_index=False).agg({
    'date': 'max',
    'num_25': 'sum',
    'num_50': 'sum',
    'num_75': 'sum',
    'num_985': 'sum',
    'num_100': 'sum',
    'num_unq': 'max',
    'total_secs': 'sum',
})

# calculate the percentage of number of songs played within certain period
song_total = (user_logs_v2['num_25'] + user_logs_v2['num_50'] + user_logs_v2['num_75']
              + user_logs_v2['num_985'] + user_logs_v2['num_100'])
user_logs_v2['percent_25'] = user_logs_v2['num_25'] / song_total
user_logs_v2['percent_50'] = user_logs_v2['num_50'] / song_total
user_logs_v2['percent_100'] = (user_logs_v2['num_985'] + user_logs_v2['num_100']) / song_total

# drop useless variables
user_logs_v3 = user_logs_v2.drop(columns=['num_25', 'num_50', 'num_75', 'num_985', 'num_100'])
user_logs_v3.head()

## 3. Merge into a Single Modeling Table

In [ ]:
dataset_train = (train
    .merge(members, on='msno', how='left')
    .merge(transactions_v2, on='msno', how='left')
    .merge(user_logs_v3, on='msno', how='left'))
dataset_train.dtypes

## 4. Parse Dates and Engineer "Days Since" Features

Dates come in from CSV as `YYYYMMDD` integers, not real dates, so they need explicit
parsing. From there we derive day-count features relative to the latest observed
expiry date — the kind of recency feature that matters a lot for churn.

In [ ]:
# date in csv will be recognized as float in python
# this value needs to be converted back to date
dataset_train['registration_init_time'] = pd.to_datetime(dataset_train['registration_init_time'], format='%Y%m%d')
dataset_train['transaction_date'] = pd.to_datetime(dataset_train['transaction_date'], format='%Y%m%d')
dataset_train['membership_expire_date'] = pd.to_datetime(dataset_train['membership_expire_date'], format='%Y%m%d')
dataset_train['date'] = pd.to_datetime(dataset_train['date'], format='%Y%m%d')

# check the maximum of datetime value
dataset_train.select_dtypes(include=['datetime64[ns]']).max()

In [ ]:
# create new day columns for modelling purpose
# NOTE: original used .astype('timedelta64[D]'), removed in modern pandas -> use .dt.days instead
latest_expiry = dataset_train['membership_expire_date'].max()
dataset_train['registration_day'] = (latest_expiry - dataset_train['registration_init_time']).dt.days
dataset_train['transaction_day'] = (latest_expiry - dataset_train['transaction_date']).dt.days
dataset_train['membership_expire_day'] = (latest_expiry - dataset_train['membership_expire_date']).dt.days
dataset_train['last_play_day'] = (latest_expiry - dataset_train['date']).dt.days

dataset_train[['registration_day', 'transaction_day', 'membership_expire_day', 'last_play_day']].describe()

In [ ]:
# check the distribution of age due to the documentation explanation
dataset_train['bd'].value_counts().head(15)

In [ ]:
import seaborn as sns
# Registration method count
city_order = sorted(members["registered_via"].unique())

plt.figure(figsize=(15,5))
sns.countplot(x="registered_via", data=members, order=city_order)
plt.ylabel("Count", fontsize=12)
plt.xlabel("Registration method code", fontsize=12)
plt.xticks(rotation='vertical')
plt.title("Users' registration methods frequencies", fontsize=12)
plt.show()

In [ ]:
# City count
city_order = sorted(members['city'].unique())

plt.figure(figsize=(15,5))
sns.countplot(x="city", data=members, order = city_order)
plt.ylabel("Count", fontsize=12)
plt.xlabel("City code", fontsize=12)
plt.xticks(rotation='vertical')
plt.title("Users' cities frequencies", fontsize=12)
plt.show()

## 5. Drop Unreliable Columns

`gender` and `bd` (age) are dropped: in the real dataset both have very high rates of
missing or garbage values (documented in the competition data notes — negative ages,
ages in the thousands, etc.), so they're excluded rather than imputed.

In [ ]:
dataset_train_v2 = dataset_train.drop(columns=[
    'msno', 'gender', 'bd', 'registration_init_time', 'transaction_date',
    'membership_expire_date', 'date'
])
dataset_train_v2.dtypes

## 6. Handle Missing Values

In [ ]:
# check the number of missing values in each column
dataset_train_v2.isna().sum()

In [ ]:
# Handle missing value of part of numeric/categorical columns by using mode
def replacemode(col):
    dataset_train_v2[col] = dataset_train_v2[col].fillna(dataset_train_v2[col].value_counts().index[0])

for col in ['city', 'registered_via', 'payment_method_id', 'payment_plan_days', 'is_auto_renew', 'is_cancel']:
    replacemode(col)

In [ ]:
# Handle missing value of part of numeric columns by using mean
from sklearn.impute import SimpleImputer
mean_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

def replacemean(col):
    dataset_train_v2[col] = mean_imputer.fit_transform(dataset_train_v2[[col]])

for col in ['plan_list_price', 'actual_amount_paid', 'num_unq', 'total_secs',
            'percent_25', 'percent_50', 'percent_100',
            'registration_day', 'transaction_day', 'membership_expire_day', 'last_play_day']:
    replacemean(col)

dataset_train_v2.isna().sum()

## 7. Cap Outliers (3-sigma winsorizing)

In [ ]:
# Handle outliers by using capping
# NOTE: original used df[col][mask] = value (chained assignment, deprecated in modern
# pandas) -> use .loc[mask, col] = value instead, which is the correct, non-chained form
def replaceoutlier(col):
    mean, std = np.mean(dataset_train_v2[col]), np.std(dataset_train_v2[col])
    cut_off = std * 3
    lower, upper = mean - cut_off, mean + cut_off
    dataset_train_v2.loc[dataset_train_v2[col] < lower, col] = lower
    dataset_train_v2.loc[dataset_train_v2[col] > upper, col] = upper

for col in ['plan_list_price', 'actual_amount_paid', 'num_unq', 'total_secs',
            'percent_25', 'percent_50', 'percent_100',
            'registration_day', 'transaction_day', 'membership_expire_day', 'last_play_day']:
    replaceoutlier(col)

dataset_train_v2.describe()

## 8. Target-Mean Encode Categorical Features

`city`, `registered_via`, and `payment_method_id` are high-cardinality categorical
codes. Rather than one-hot encoding them, each is replaced with its historical churn
rate — a form of target encoding.

In [ ]:
# convert categorical variables into string first (so groupby treats them as categories, not numbers)
cat_cols = ['city', 'registered_via', 'payment_method_id']
dataset_train_v2[cat_cols] = dataset_train_v2[cat_cols].astype(str)

# replace discrete features with historical churn rate
city_mean = dataset_train_v2.groupby('city')['is_churn'].mean().reset_index(name='city_mean')
register_mean = dataset_train_v2.groupby('registered_via')['is_churn'].mean().reset_index(name='register_mean')
payment_mean = dataset_train_v2.groupby('payment_method_id')['is_churn'].mean().reset_index(name='payment_mean')

dataset_train_v3 = (dataset_train_v2
    .merge(city_mean, on='city', how='left')
    .merge(register_mean, on='registered_via', how='left')
    .merge(payment_mean, on='payment_method_id', how='left')
    .drop(columns=['city', 'registered_via', 'payment_method_id']))

dataset_train_v3.head()

## 9. Feature Scaling

In [ ]:
# Feature Scaling for modelling purpose by using min-max-scaling
from sklearn.preprocessing import MinMaxScaler

X = dataset_train_v3.drop(columns=['is_churn'])
Y = dataset_train_v3['is_churn']

nm_X = pd.DataFrame(MinMaxScaler().fit_transform(X))
nm_X.columns = X.columns.values
nm_X.index = X.index.values
nm_X.head()

## 10. Feature Selection

Binary flags (`is_auto_renew`, `is_cancel`) are tested with chi-squared; continuous
features are tested with an ANOVA F-test. Only features with p ≤ 0.05 against the
churn target are kept.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2, f_classif

nm_col = ['is_auto_renew', 'is_cancel']
nm_X_v2 = nm_X.drop(columns=nm_col)     # continuous features
nm_X_v3 = pd.DataFrame(nm_X, columns=nm_col)  # binary features

nm_X_v4 = pd.DataFrame(
    SelectKBest(score_func=chi2, k='all').fit(nm_X_v3, Y).pvalues_ <= 0.05,
    columns=['importance'])
nm_X_v4.index = nm_X_v3.columns.values

nm_X_v5 = pd.DataFrame(
    SelectKBest(score_func=f_classif, k='all').fit(nm_X_v2, Y).pvalues_ <= 0.05,
    columns=['importance'])
nm_X_v5.index = nm_X_v2.columns.values

nm_X_v6 = pd.concat([nm_X_v4, nm_X_v5])
nm_selected = list(pd.Series(nm_X_v6[nm_X_v6['importance'] == 1].index.values))
nm_X_v7 = pd.DataFrame(nm_X, columns=nm_selected)

print('Selected features:', nm_selected)

## 11. Dimension Reduction (PCA)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit_transform(nm_X_v7)
print('Cumulative explained variance:')
print(np.round(np.cumsum(pca.explained_variance_ratio_), 3))

nm_X_v8 = PCA(n_components=10).fit_transform(nm_X_v7)

## 12. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

nm_X_train, nm_X_test, nm_Y_train, nm_Y_test = train_test_split(
    nm_X_v8, Y, test_size=0.3, random_state=0)

print(f"train: {nm_X_train.shape}, test: {nm_X_test.shape}, churn rate train: {nm_Y_train.mean():.3f}")

## 13. Model Comparison

10-fold cross-validated log-loss across a battery of classifiers and ensembles.

**Compatibility fix:** `BaggingClassifier`/`AdaBoostClassifier` renamed their base-model
argument from `base_estimator` to `estimator` in scikit-learn 1.2+ — updated below.

**Robustness fix:** the original loop had no error handling, so a single model's
numerical failure (e.g. QDA hitting a singular covariance matrix, a real and fairly
common edge case with correlated features) would kill the whole cell. Wrapped in
try/except so one failure doesn't block the rest.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier,
                               RandomForestClassifier, ExtraTreesClassifier)
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

nm_models = [
    ('KNN', KNeighborsClassifier()),
    ('LR', LogisticRegression()),
    ('LDA', LinearDiscriminantAnalysis()),
    ('QDA', QuadraticDiscriminantAnalysis()),
    ('CART', DecisionTreeClassifier()),
    ('NB', GaussianNB()),
    # ('Linear SVM', SVC(kernel='linear')),
    # ('Kernel SVM', SVC(kernel='rbf')),
]

ensembles = [
    ('BC', BaggingClassifier(estimator=LogisticRegression())),
    ('AB', AdaBoostClassifier(estimator=LogisticRegression())),
    ('GBM', GradientBoostingClassifier()),
    ('RF', RandomForestClassifier()),
    ('ET', ExtraTreesClassifier()),
    ('XGB', XGBClassifier()),
]

In [ ]:
results, names = [], []
for name, model in nm_models:
    try:
        cv_results = cross_val_score(model, nm_X_train, nm_Y_train, cv=10,
                                      scoring='neg_log_loss', n_jobs=-1, error_score='raise')
        results.append(cv_results)
        names.append(name)
        print(f"{name}: {cv_results.mean():.6f} ({cv_results.std():.6f})")
    except Exception as e:
        print(f"{name}: FAILED - {type(e).__name__}: {str(e)[:120]}")

In [ ]:
results2, names2 = [], []
for name2, model2 in ensembles:
    try:
        cv_results2 = cross_val_score(model2, nm_X_train, nm_Y_train, cv=10,
                                       scoring='neg_log_loss', n_jobs=-1, error_score='raise')
        results2.append(cv_results2)
        names2.append(name2)
        print(f"{name2}: {cv_results2.mean():.6f} ({cv_results2.std():.6f})")
    except Exception as e:
        print(f"{name2}: FAILED - {type(e).__name__}: {str(e)[:120]}")

## 14. Hyperparameter Tuning: XGBoost

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
    "learning_rate": [0.1],
    "min_child_weight": [1],
    "max_depth": [9],
    "gamma": [0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "objective": ['binary:logistic'],
}
grid_search_XGB = GridSearchCV(estimator=XGBClassifier(), param_grid=parameters,
                                scoring="neg_log_loss", cv=10, n_jobs=-1)
grid_result_XGB = grid_search_XGB.fit(nm_X_train, nm_Y_train)
print(f"Best: {grid_result_XGB.best_score_:.6f} using {grid_result_XGB.best_params_}")

## 15. Final Evaluation on Held-Out Test Set

In [ ]:
from sklearn.metrics import log_loss

nm_Y_predict = grid_result_XGB.predict_proba(nm_X_test)
logloss = log_loss(nm_Y_test, nm_Y_predict)
print(f"Log Loss Score: {logloss}")

## 16. Classification Metrics, Feature Importance and Segment Diagnostics

The hold-out split is 70/30 (`test_size=0.30`, `random_state=0`). Metrics use churn=1 as the positive class and a 0.50 threshold. Important limitation: target-mean encoding, imputation, scaling, feature selection and PCA were fitted before the split in the legacy workflow. The metrics below are therefore exploratory and may be optimistic; use a leakage-safe pipeline and time-based validation before production use.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, RocCurveDisplay
)
import seaborn as sns
import joblib
from pathlib import Path

best_model = grid_result_XGB.best_estimator_
test_probability = best_model.predict_proba(nm_X_test)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

metrics = pd.Series({
    "ROC-AUC": roc_auc_score(nm_Y_test, test_probability),
    "Accuracy": accuracy_score(nm_Y_test, test_prediction),
    "Precision": precision_score(nm_Y_test, test_prediction, zero_division=0),
    "Recall": recall_score(nm_Y_test, test_prediction, zero_division=0),
    "F1": f1_score(nm_Y_test, test_prediction, zero_division=0),
    "Log-loss": log_loss(nm_Y_test, test_probability),
}, name="held_out_test")
display(metrics.to_frame())

cm = confusion_matrix(nm_Y_test, test_prediction, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted retained", "Predicted churn"],
            yticklabels=["Actual retained", "Actual churn"])
plt.title("Churn model — held-out test confusion matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()
RocCurveDisplay.from_predictions(nm_Y_test, test_probability)
plt.title("ROC curve — held-out test")
plt.tight_layout()
plt.show()
print(classification_report(nm_Y_test, test_prediction,
                            target_names=["retained", "churn"], zero_division=0))


## 17. Feature Importance and Churn Drivers

The classifier is fitted on PCA components. This cell maps component importance back to selected source features using squared PCA loadings. It is an exploratory attribution, not a causal explanation.

In [ ]:
component_importance = pd.Series(
    best_model.feature_importances_,
    index=[f"PC{i+1}" for i in range(len(best_model.feature_importances_))]
)
loadings = pd.DataFrame(
    pca.components_[:len(component_importance), :],
    columns=nm_X_v7.columns,
    index=component_importance.index
)
mapped_importance = loadings.pow(2).mul(component_importance, axis=0).sum(axis=0)
mapped_importance = mapped_importance.sort_values(ascending=False)
display(mapped_importance.rename("mapped_importance").to_frame())
mapped_importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Exploratory feature importance mapped from PCA components")
plt.xlabel("Weighted importance")
plt.tight_layout()
plt.show()


## 18. Churn by Customer Segment

The following segment rates are descriptive summaries of the labelled sample, not causal effects. Usage level is based on quartiles of total listening seconds.

In [ ]:
segment_data = dataset_train[["is_churn", "payment_plan_days", "is_auto_renew",
                             "city", "total_secs"]].copy()
segment_data["plan_segment"] = segment_data["payment_plan_days"].fillna(-1).astype(str)
segment_data["auto_renew_segment"] = segment_data["is_auto_renew"].fillna(-1).astype(str)
segment_data["city_segment"] = segment_data["city"].fillna(-1).astype(str)
segment_data["usage_level"] = pd.qcut(
    segment_data["total_secs"].rank(method="first"),
    q=4, labels=["Low", "Medium-low", "Medium-high", "High"]
)
for dimension, column in {
    "Plan": "plan_segment", "Auto-renew": "auto_renew_segment",
    "City": "city_segment", "Usage": "usage_level"
}.items():
    summary = (segment_data.groupby(column, observed=False)["is_churn"]
               .agg(customers="size", churn_rate="mean")
               .sort_values("churn_rate", ascending=False))
    print(f"\n{dimension} segment churn")
    display(summary)


## 19. Save Research Model Artifact

This artifact is for reproducibility of the notebook experiment only. It does not package all upstream preprocessing as a production inference pipeline; do not deploy it until preprocessing leakage is removed and the complete transform chain is serialized and validated.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
joblib.dump({
    "model": best_model,
    "pca_components": pca.components_[:10],
    "selected_features": list(nm_X_v7.columns),
    "threshold": 0.50,
    "target": "is_churn",
    "split": {"test_size": 0.30, "random_state": 0},
}, artifact_dir / "kkbox_xgboost_churn.joblib")
print("Saved:", artifact_dir / "kkbox_xgboost_churn.joblib")
